# Wyoming 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Wyoming, 2008 for presidential general election results, and then derive summary stats (party totals). Note, there is no presidential primary election results for Arkansas 2008 so far.

**Output**: A single CSV where each row is a county and columns include:

- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_general_total`, `dem_general_total`, `lib_general_total`, `ind_general_total`

**Last Updated**: 2025/10/11

## 0. Library Import

In [55]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Inputs & Parameters

Define raw file paths once here so the entire notebook is easy to rerun on another machine. If a path changes, we only update it here. We keep a single `OUTPUT_PATH` so all exports land in one known place.

In [56]:
# WY 2008 dataset path
PRIMARY_PATH = r"../../data/raw/2008/WY/20080819__wy__primary__precinct__raw.csv"
GENERAL_PATH = r"../../data/raw/2008/WY/20081104__wy__general__precinct__raw.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/WY/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

- Further any additional complications at each step

### a. Primary Election Dataset

In [57]:
# Load primary data
primary_df = pd.read_csv(PRIMARY_PATH)
primary_df.head(DISPLAY_ROWS)

,updated_at,id,start_date,end_date,election_type,result_type,special,office,district,name_raw,...,party,parent_jurisdiction,jurisdiction,division,votes,votes_type,total_votes,winner,write_in,year
0,2015-10-10 10:38:21.751000,wy-2008-08-19-primary,2008-08-19 00:00:00,2008-08-19 00:00:00,primary,certified,False,United States Senator,NaN,Al Hamburg,...,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,11,NaN,NaN,NaN,NaN,2008
1,2015-10-10 10:38:21.751000,wy-2008-08-19-primary,2008-08-19 00:00:00,2008-08-19 00:00:00,primary,certified,False,United States Senator,NaN,Chris Rothfuss,...,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,34,NaN,NaN,NaN,NaN,2008
2,2015-10-10 10:38:21.751000,wy-2008-08-19-primary,2008-08-19 00:00:00,2008-08-19 00:00:00,primary,certified,False,State United,4.0,Nick Carter,...,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,21,NaN,NaN,NaN,NaN,2008
3,2015-10-10 10:38:21.751000,wy-2008-08-19-primary,2008-08-19 00:00:00,2008-08-19 00:00:00,primary,certified,False,State United,4.0,Keith B. Goodenough,...,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,23,NaN,NaN,NaN,NaN,2008
4,2015-10-10 10:38:21.751000,wy-2008-08-19-primary,2008-08-19 00:00:00,2008-08-19 00:00:00,primary,certified,False,United States Representative,NaN,Gary Trauner,...,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,43,NaN,NaN,NaN,NaN,2008
5,2015-10-10 10:38:21.751000,wy-2008-08-19-primary,2008-08-19 00:00:00,2008-08-19 00:00:00,primary,certified,False,State House,14.0,Pat Kiovsky,...,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,36,NaN,NaN,NaN,NaN,2008
6,2015-10-10 10:38:21.751000,wy-2008-08-19-primary,2008-08-19 00:00:00,2008-08-19 00:00:00,primary,certified,False,United States Senator,NaN,Al Hamburg,...,Democratic,Albany,Albany County Fairgrounds/Harmony 12-1,ocd-division/country:us/state:wy/county:albany...,50,NaN,NaN,NaN,NaN,2008
7,2015-10-10 10:38:21.751000,wy-2008-08-19-primary,2008-08-19 00:00:00,2008-08-19 00:00:00,primary,certified,False,United States Senator,NaN,Chris Rothfuss,...,Democratic,Albany,Albany County Fairgrounds/Harmony 12-1,ocd-division/country:us/state:wy/county:albany...,130,NaN,NaN,NaN,NaN,2008
8,2015-10-10 10:38:21.751000,wy-2008-08-19-primary,2008-08-19 00:00:00,2008-08-19 00:00:00,primary,certified,False,State United,4.0,Nick Carter,...,Democratic,Albany,Albany County Fairgrounds/Harmony 12-1,ocd-division/country:us/state:wy/county:albany...,93,NaN,NaN,NaN,NaN,2008
9,2015-10-10 10:38:21.751000,wy-2008-08-19-primary,2008-08-19 00:00:00,2008-08-19 00:00:00,primary,certified,False,State United,4.0,Keith B. Goodenough,...,Democratic,Albany,Albany County Fairgrounds/Harmony 12-1,ocd-division/country:us/state:wy/county:albany...,97,NaN,NaN,NaN,NaN,2008


In [58]:
# Data shape
primary_df.shape

(6710, 24)

In [59]:
# Missing values count
primary_df.isnull().sum()

updated_at                0
id                        0
start_date                0
end_date                  0
election_type             0
result_type               0
special                   0
office                    0
district               4259
name_raw                  0
last_name              6710
first_name             6710
suffix                 6710
middle_name            6710
party                     0
parent_jurisdiction       0
jurisdiction              0
division                  0
votes                     0
votes_type             6710
total_votes            6710
winner                 6710
write_in               6710
year                      0
dtype: int64

There are multiple columns where they are all just missing data. Also, some columns are just identifiers (`updated_at`, `id`, `start_date`, `end_date`) or repeated values (`year`). We will first drop all of these and further investigate the rest.

In [60]:
# Drop unnecessary columns
primary_df = primary_df.drop(
    columns=["updated_at", "id", "start_date", "end_date", 
             "last_name", "first_name", "suffix", "middle_name", 
             "votes_type", "total_votes", "winner", "write_in", "year"]
    ).reset_index(drop=True)

# Snippet of the subset of the data that we want to keep
primary_df.head(DISPLAY_ROWS)

,election_type,result_type,special,office,district,name_raw,party,parent_jurisdiction,jurisdiction,division,votes
0,primary,certified,False,United States Senator,NaN,Al Hamburg,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,11
1,primary,certified,False,United States Senator,NaN,Chris Rothfuss,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,34
2,primary,certified,False,State United,4.0,Nick Carter,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,21
3,primary,certified,False,State United,4.0,Keith B. Goodenough,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,23
4,primary,certified,False,United States Representative,NaN,Gary Trauner,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,43
5,primary,certified,False,State House,14.0,Pat Kiovsky,Democratic,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,36
6,primary,certified,False,United States Senator,NaN,Al Hamburg,Democratic,Albany,Albany County Fairgrounds/Harmony 12-1,ocd-division/country:us/state:wy/county:albany...,50
7,primary,certified,False,United States Senator,NaN,Chris Rothfuss,Democratic,Albany,Albany County Fairgrounds/Harmony 12-1,ocd-division/country:us/state:wy/county:albany...,130
8,primary,certified,False,State United,4.0,Nick Carter,Democratic,Albany,Albany County Fairgrounds/Harmony 12-1,ocd-division/country:us/state:wy/county:albany...,93
9,primary,certified,False,State United,4.0,Keith B. Goodenough,Democratic,Albany,Albany County Fairgrounds/Harmony 12-1,ocd-division/country:us/state:wy/county:albany...,97


Now, we have a better look of our dataset. We can first see that we could further drop `division`. Also, `election_type`, `result_type`, and `special` look like something we could drop. But first, we will check the different values in these columns.

In [61]:
# Drop 'division' column
primary_df = primary_df.drop(columns=["division"]).reset_index(drop=True)
primary_df.shape

(6710, 10)

In [62]:
# Number of unique values in all columns
{col: primary_df[col].nunique() for col in primary_df.columns}

{'election_type': 1,
 'result_type': 1,
 'special': 1,
 'office': 7,
 'district': 49,
 'name_raw': 117,
 'party': 3,
 'parent_jurisdiction': 23,
 'jurisdiction': 488,
 'votes': 363}

So, all of the three columns that we were suspected all have a single value. These columns will not contribute any meaning to our analysis. Thus, we can just drop it.

In [63]:
# Drop 'election_type', 'result_type', and 'special' columns
primary_df = primary_df.drop(columns=["election_type", "result_type", "special"]).reset_index(drop=True)
primary_df.shape

(6710, 7)

Also, we only want data of the presidential election. Thus, we would filter the observations out and only take the related observations.

In [64]:
# Number of unique values in all columns
{col: primary_df[col].nunique() for col in primary_df.columns}

{'office': 7,
 'district': 49,
 'name_raw': 117,
 'party': 3,
 'parent_jurisdiction': 23,
 'jurisdiction': 488,
 'votes': 363}

In [65]:
# Difference values in 'office' column
primary_df["office"].value_counts()

office
United States Representative     2598
State United                     1458
United States Senator            1413
State House                       609
State Senate                      384
United States  Representative     203
United States  Senator             45
Name: count, dtype: int64

But then, there are no presidential election data in Wyoming primary dataset. We can stop it here for now, and maybe pick up later if we need to.

### b. General Election Dataset

In [66]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,updated_at,id,start_date,end_date,election_type,result_type,special,office,district,name_raw,...,party,parent_jurisdiction,jurisdiction,division,votes,votes_type,total_votes,winner,write_in,year
0,2015-10-10 10:37:50.348000,wy-2008-11-04-general,2008-11-04 00:00:00,2008-11-04 00:00:00,general,certified,False,United States President and Vice President,NaN,John McCain and Sarah Palin,...,R,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,364,NaN,NaN,NaN,NaN,2008
1,2015-10-10 10:37:50.348000,wy-2008-11-04-general,2008-11-04 00:00:00,2008-11-04 00:00:00,general,certified,False,United States President and Vice President,NaN,Barack Obama and Joe Biden,...,D,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,204,NaN,NaN,NaN,NaN,2008
2,2015-10-10 10:37:50.348000,wy-2008-11-04-general,2008-11-04 00:00:00,2008-11-04 00:00:00,general,certified,False,United States President and Vice President,NaN,Bob Barr and Wayne A. Root,...,L,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,4,NaN,NaN,NaN,NaN,2008
3,2015-10-10 10:37:50.348000,wy-2008-11-04-general,2008-11-04 00:00:00,2008-11-04 00:00:00,general,certified,False,United States President and Vice President,NaN,Chuck Baldwin and Darrell L.Castle,...,I,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,0,NaN,NaN,NaN,NaN,2008
4,2015-10-10 10:37:50.348000,wy-2008-11-04-general,2008-11-04 00:00:00,2008-11-04 00:00:00,general,certified,False,United States President and Vice President,NaN,Ralph Nader and Matt Gonzalez,...,I,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,2,NaN,NaN,NaN,NaN,2008
5,2015-10-10 10:37:50.348000,wy-2008-11-04-general,2008-11-04 00:00:00,2008-11-04 00:00:00,general,certified,False,United States President and Vice President,NaN,Write-Ins,...,NaN,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,3,NaN,NaN,NaN,NaN,2008
6,2015-10-10 10:37:50.348000,wy-2008-11-04-general,2008-11-04 00:00:00,2008-11-04 00:00:00,general,certified,False,United States President and Vice President,NaN,Under Votes,...,NaN,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,3,NaN,NaN,NaN,NaN,2008
7,2015-10-10 10:37:50.348000,wy-2008-11-04-general,2008-11-04 00:00:00,2008-11-04 00:00:00,general,certified,False,United States President and Vice President,NaN,Over Votes,...,NaN,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,0,NaN,NaN,NaN,NaN,2008
8,2015-10-10 10:37:50.348000,wy-2008-11-04-general,2008-11-04 00:00:00,2008-11-04 00:00:00,general,certified,False,United States Senator,NaN,Mike Enzi,...,R,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,414,NaN,NaN,NaN,NaN,2008
9,2015-10-10 10:37:50.348000,wy-2008-11-04-general,2008-11-04 00:00:00,2008-11-04 00:00:00,general,certified,False,United States Senator,NaN,Chris Rothfuss,...,D,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,152,NaN,NaN,NaN,NaN,2008


In [67]:
# Different columns in general_df
general_df.columns

Index(['updated_at', 'id', 'start_date', 'end_date', 'election_type',
       'result_type', 'special', 'office', 'district', 'name_raw', 'last_name',
       'first_name', 'suffix', 'middle_name', 'party', 'parent_jurisdiction',
       'jurisdiction', 'division', 'votes', 'votes_type', 'total_votes',
       'winner', 'write_in', 'year'],
      dtype='object')

Again, we will only keep columns that we are really needed. We can first drop `updated_at`, `id`, `start_date`, `end_date` and further investigate if we can drop any other.

In [68]:
# Drop 'updated_at', 'id', 'start_date', 'end_date' columns
general_df = general_df.drop(columns=["updated_at", "id", "start_date", "end_date"]).reset_index(drop=True)
general_df.shape

(14974, 20)

In [69]:
# Check for missing values of different columns left in general_df
general_df.isnull().sum()

election_type              0
result_type                0
special                    0
office                     0
district                9234
name_raw                   0
last_name              14974
first_name             14974
suffix                 14974
middle_name            14974
party                   8037
parent_jurisdiction        0
jurisdiction               0
division                   0
votes                      0
votes_type             14974
total_votes            14974
winner                 14974
write_in               14974
year                       0
dtype: int64

Again, we can drop columns that don't have any values, which includes `last_name`, `first_name`, `suffix`, `middle_name`, `votes_type`, `total_votes`, `winner` and `write_in`.

In [70]:
# Drop columns that only have missing values
general_df = general_df.drop(columns=["last_name", "first_name", "suffix", "middle_name", "total_votes", "votes_type", "winner", "write_in"]).reset_index(drop=True)
general_df.shape

(14974, 12)

In [71]:
# Quick look of the current state of general_df
general_df.head(DISPLAY_ROWS)

,election_type,result_type,special,office,district,name_raw,party,parent_jurisdiction,jurisdiction,division,votes,year
0,general,certified,False,United States President and Vice President,NaN,John McCain and Sarah Palin,R,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,364,2008
1,general,certified,False,United States President and Vice President,NaN,Barack Obama and Joe Biden,D,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,204,2008
2,general,certified,False,United States President and Vice President,NaN,Bob Barr and Wayne A. Root,L,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,4,2008
3,general,certified,False,United States President and Vice President,NaN,Chuck Baldwin and Darrell L.Castle,I,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,0,2008
4,general,certified,False,United States President and Vice President,NaN,Ralph Nader and Matt Gonzalez,I,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,2,2008
5,general,certified,False,United States President and Vice President,NaN,Write-Ins,NaN,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,3,2008
6,general,certified,False,United States President and Vice President,NaN,Under Votes,NaN,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,3,2008
7,general,certified,False,United States President and Vice President,NaN,Over Votes,NaN,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,0,2008
8,general,certified,False,United States Senator,NaN,Mike Enzi,R,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,414,2008
9,general,certified,False,United States Senator,NaN,Chris Rothfuss,D,Albany,Northern Albany County 11-1,ocd-division/country:us/state:wy/county:albany...,152,2008


By looking at it, we can see that we can also drop `division` and `yaer` columns. The first three columns also looks like we can drop it. Yet, we will check the number of unique values in those columns to make sure that we don't drop any important information.

In [72]:
# Number of unique values in each column
{col: general_df[col].nunique() for col in general_df.columns}

{'election_type': 1,
 'result_type': 1,
 'special': 1,
 'office': 6,
 'district': 61,
 'name_raw': 128,
 'party': 4,
 'parent_jurisdiction': 23,
 'jurisdiction': 486,
 'division': 486,
 'votes': 888,
 'year': 1}

In [73]:
# Further drop columns that are unnecessary
general_df = general_df.drop(columns=["election_type", "result_type", "special", "division", "year"]).reset_index(drop=True)
general_df.shape

(14974, 7)

We now check the `office` column to see that there is presidential election data in such dataframe since we are interested in the presidential election result. If yes, we will filter to only keep those rows for further cleaning and investigation.

In [74]:
# Different values in 'office' column
general_df["office"].value_counts()

office
United States President and Vice President    3888
United States Representative                  2916
United States Senator                         2430
State United                                  2430
State House                                   2136
State Senate                                  1174
Name: count, dtype: int64

In [75]:
# Only keep rows where 'office' is 'United States President and Vice President'
general_df = general_df[general_df["office"] == "United States President and Vice President"]
general_df.head(DISPLAY_ROWS)

,office,district,name_raw,party,parent_jurisdiction,jurisdiction,votes
0,United States President and Vice President,NaN,John McCain and Sarah Palin,R,Albany,Northern Albany County 11-1,364
1,United States President and Vice President,NaN,Barack Obama and Joe Biden,D,Albany,Northern Albany County 11-1,204
2,United States President and Vice President,NaN,Bob Barr and Wayne A. Root,L,Albany,Northern Albany County 11-1,4
3,United States President and Vice President,NaN,Chuck Baldwin and Darrell L.Castle,I,Albany,Northern Albany County 11-1,0
4,United States President and Vice President,NaN,Ralph Nader and Matt Gonzalez,I,Albany,Northern Albany County 11-1,2
5,United States President and Vice President,NaN,Write-Ins,NaN,Albany,Northern Albany County 11-1,3
6,United States President and Vice President,NaN,Under Votes,NaN,Albany,Northern Albany County 11-1,3
7,United States President and Vice President,NaN,Over Votes,NaN,Albany,Northern Albany County 11-1,0
34,United States President and Vice President,NaN,John McCain and Sarah Palin,R,Albany,Albany County Fairgrounds/Harmony 12-1,1230
35,United States President and Vice President,NaN,Barack Obama and Joe Biden,D,Albany,Albany County Fairgrounds/Harmony 12-1,742


In [76]:
# Unique values in 'parent_jurisdiction'
general_df["parent_jurisdiction"].value_counts()

parent_jurisdiction
Laramie        480
Natrona        368
Sweetwater     288
Campbell       280
Fremont        256
Sublette       232
Park           232
Albany         176
Goshen         160
Carbon         152
Crook          152
Teton          144
Lincoln        144
Johnson        136
Converse       136
Big Horn       104
Platte         104
Uinta           88
Sheridan        72
Weston          64
Niobrara        48
Washakie        40
Hot Springs     32
Name: count, dtype: int64

In [77]:
# Number of missing values in each column now
general_df.isna().sum()

office                    0
district               3888
name_raw                  0
party                  1458
parent_jurisdiction       0
jurisdiction              0
votes                     0
dtype: int64

We can see that the 23 different values in the `parent_jurisdiction` actually match with 23 different counties in Wyoming. Given that we are interested in county-level data for presidential election, we can just keep this column, rename it to `county`, and drop `district` since it is just empty now. We can further drop `jurisdiction` column too as it does not provide much additional information that we need.

In [79]:
# Drop 'district' and 'jurisdiction' column
general_df = general_df.drop(columns=["district", "jurisdiction"]).reset_index(drop=True)
general_df.shape

(3888, 5)

In [80]:
# Rename 'parent_jurisdiction' to 'county' and "name_raw" to "candidate"
general_df = general_df.rename(columns={"parent_jurisdiction": "county", "name_raw": "candidate"})

# Rearrange columns
general_df = general_df[["county", "candidate", "party", "votes"]]
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Albany,John McCain and Sarah Palin,R,364
1,Albany,Barack Obama and Joe Biden,D,204
2,Albany,Bob Barr and Wayne A. Root,L,4
3,Albany,Chuck Baldwin and Darrell L.Castle,I,0
4,Albany,Ralph Nader and Matt Gonzalez,I,2
5,Albany,Write-Ins,NaN,3
6,Albany,Under Votes,NaN,3
7,Albany,Over Votes,NaN,0
8,Albany,John McCain and Sarah Palin,R,1230
9,Albany,Barack Obama and Joe Biden,D,742


There are something that we can notice here. First, the values in candidate now include both president candidate and vice president candidate. Thus, we will need to clean this and only keep presidential candidate. Second, there are some unrelated values like "Write-Ins", "Under Votes", or "Over Votes". We don't want to keep this in our dataframe. Thus, we will remove them and anything similar to that.

In [81]:
# Different values in 'candidate' column
general_df["candidate"].value_counts()

candidate
John McCain  and Sarah Palin           486
Barack Obama and  Joe Biden            486
Bob Barr and  Wayne A. Root            486
Chuck Baldwin and  Darrell L.Castle    486
Ralph Nader and  Matt Gonzalez         486
Write-Ins                              486
Under Votes                            486
Over Votes                             486
Name: count, dtype: int64

In [83]:
# Drop observations where values in 'candidate' column are in ["Write-Ins", "Under Votes", Over Votes]
general_df = general_df.loc[~ general_df["candidate"].isin(
    ["Write-Ins", "Under Votes", "Over Votes"]
)].copy()

general_df.shape

(2430, 4)

In [84]:
# Split the names in 'candidate' column and only keep presidential candidate
general_df["candidate"] = (
    general_df["candidate"].astype(str).str.strip()
    .str.split("and").str[0]
)

# Snippet of the dataframe after cleaning
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Albany,John McCain,R,364
1,Albany,Barack Obama,D,204
2,Albany,Bob Barr,L,4
3,Albany,Chuck Baldwin,I,0
4,Albany,Ralph Nader,I,2
8,Albany,John McCain,R,1230
9,Albany,Barack Obama,D,742
10,Albany,Bob Barr,L,15
11,Albany,Chuck Baldwin,I,10
12,Albany,Ralph Nader,I,28


In [85]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
I    972
R    486
D    486
L    486
Name: count, dtype: int64

In [86]:
# Missing values count
general_df.isnull().sum()

county       0
candidate    0
party        0
votes        0
dtype: int64

In [87]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Albany,John McCain,R,364
1,Albany,Barack Obama,D,204
2,Albany,Bob Barr,L,4
3,Albany,Chuck Baldwin,I,0
4,Albany,Ralph Nader,I,2
8,Albany,John McCain,R,1230
9,Albany,Barack Obama,D,742
10,Albany,Bob Barr,L,15
11,Albany,Chuck Baldwin,I,10
12,Albany,Ralph Nader,I,28


In [88]:
# Shape after preprocessing
general_df.shape

(2430, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [90]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "D"     : "dem", 
                "R"     : "rep",
                "L"     : "lib",
                "I"     : "ind",
               })
           .fillna(s.str.strip().str.lower()))      # For defensive purposes only, would not expect other parties

In [91]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [92]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [93]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_ind_BALDWIN,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
0,Albany,8644,64,201,149,7936
1,Big Horn,1108,39,41,37,4045
2,Campbell,2990,57,105,77,13011
3,Carbon,2336,22,81,46,4331
4,Converse,612,22,31,29,2967
5,Crook,1380,28,66,27,4922
6,Fremont,6016,91,190,102,11083
7,Goshen,1832,31,51,26,3942
8,Hot Springs,619,15,27,28,1834
9,Johnson,908,20,33,31,3334


## 4. Adding Party Total Columns

Now, we will add party totals columns for general totals:

* `rep_general_total` = sum of all `gen_rep_*` columns
* `dem_general_total` = sum of all `gen_dem_*` columns
* `lib_general_total` = sum of all `gen_lib_*` columns
* `ind_general_total` = sum of all `gen_ind_*` columns

In [94]:
# Add party totals for general election
rep_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_rep")] 
dem_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_dem")]
lib_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_lib")]
ind_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_ind")]


general_pivot["rep_general_total"] = general_pivot[rep_general_cols].sum(axis=1) if rep_general_cols else 0
general_pivot["dem_general_total"] = general_pivot[dem_general_cols].sum(axis=1) if dem_general_cols else 0
general_pivot["lib_general_total"] = general_pivot[lib_general_cols].sum(axis=1) if lib_general_cols else 0
general_pivot["ind_general_total"] = general_pivot[ind_general_cols].sum(axis=1) if ind_general_cols else 0

In [95]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned general dataframe:")
general_pivot.columns

Final columns in the cleaned general dataframe:


Index(['county', 'gen_dem_OBAMA', 'gen_ind_BALDWIN', 'gen_ind_NADER',
       'gen_lib_BARR', 'gen_rep_MCCAIN', 'rep_general_total',
       'dem_general_total', 'lib_general_total', 'ind_general_total'],
      dtype='object')

In [96]:
# Preview the general_pivot dataframe with totals
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_ind_BALDWIN,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN,rep_general_total,dem_general_total,lib_general_total,ind_general_total
0,Albany,8644,64,201,149,7936,7936,8644,149,265
1,Big Horn,1108,39,41,37,4045,4045,1108,37,80
2,Campbell,2990,57,105,77,13011,13011,2990,77,162
3,Carbon,2336,22,81,46,4331,4331,2336,46,103
4,Converse,612,22,31,29,2967,2967,612,29,53
5,Crook,1380,28,66,27,4922,4922,1380,27,94
6,Fremont,6016,91,190,102,11083,11083,6016,102,281
7,Goshen,1832,31,51,26,3942,3942,1832,26,82
8,Hot Springs,619,15,27,28,1834,1834,619,28,42
9,Johnson,908,20,33,31,3334,3334,908,31,53


In [97]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
general_pivot.to_csv(OUTPUT_PATH + "WY.csv", index=False)